# CTDDG · BindingDB preprocessing for the IIT Palakkad repository

**Based on the CTDDG paper (Sections 2.1–2.4) and the authors’ `data_preprocessing_1.ipynb`.**

This version is adapted to:

```python
data_path = '/home/muhamed/repo/CTDDG/datasets/data/bindingdb/'
```

It reads all five raw Grechishnikova BindingDB splits, reconstructs the Dataset 1 **1,042 training / 104 held-out protein** cohort deterministically, keeps ligands paired to their proteins while filtering, prevents train/test protein overlap, and writes outputs **outside** the source folders.

**Scope:** This is data preparation, **not a complete CTDDG training reproduction**. CTDDG also requires verified Ames toxicity labels (Random Forest on ChEMBL; pkCSM on BindingDB), a checked protein-embedding/training-loader contract, and its two training phases. The paper itself has inconsistent BindingDB pair counts (166,400 in §2.1, 66,400 in §2.4); these are reported, not hardcoded.

**Execution policy:** Run light CPU cells first. ChEMBL processing and ProtTransBERT embedding are optional; embeddings may need a Slurm GPU allocation. Do not execute intensive code on an HPC login node. Nothing overwrites the source files.

## 0 · Configuration and dependency check

**Required:** Python, pandas, NumPy, RDKit. **Optional:** `bio-embeddings`, `h5py` and GPU for ProtTransBERT-BFD. Installation is deliberately not automatic: use the repository's existing environment and verify compatibility. Choose a **new** `OUTPUT_DIR` for another run; writing to an existing populated output directory is blocked by default.

`SPLIT_MODE="reconstruct"` selects proteins deterministically from the supplied five raw folds. Set `SPLIT_MODE="reuse_existing"` **only if** the original Dataset 1 unique-protein files are present and contain 1042/104 disjoint proteins. In either mode, all protein–ligand pairs are rebuilt from the raw inputs.

In [ ]:
from pathlib import Path
from collections import defaultdict, Counter
from datetime import datetime, timezone
import hashlib, json, random, csv, warnings, shutil
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit import rdBase

REPO_ROOT = Path('/home/muhamed/repo/CTDDG')
data_path = '/home/muhamed/repo/CTDDG/datasets/data/bindingdb/'
DATA_PATH = Path(data_path)
CHEMBL_CSV = REPO_ROOT / 'datasets/data/chembl/chembl.csv'
OUTPUT_DIR = REPO_ROOT / 'datasets/processed/ctddg_bindingdb_v1'

RANDOM_SEED = 17
SPLIT_MODE = 'reconstruct'      # 'reconstruct' or 'reuse_existing'
TRAIN_PROTEINS_TARGET = 1042
TEST_PROTEINS_TARGET = 104
MAX_ATOMS = 100
ALLOW_OVERWRITE = False         # deliberately safe
RUN_CHEMBL_PREP = False         # optional: slow on >1M molecules
RUN_EMBEDDINGS = False          # optional: expensive; run on allocated GPU
EMBEDDING_MODE = 'residue'      # matches original embedder.embed() type; alternative: 'mean'

FOLDS = ('1042_104', '1000_112', '1004_122', '1002_103', '1036_124')
assert SPLIT_MODE in {'reconstruct', 'reuse_existing'}
assert EMBEDDING_MODE in {'residue', 'mean'}
print('RDKit:', rdBase.rdkitVersion)
print('BindingDB:', DATA_PATH)
print('Output:', OUTPUT_DIR)
print('Embedding enabled:', RUN_EMBEDDINGS, '| ChEMBL enabled:', RUN_CHEMBL_PREP)

## 1 · Inspect raw BindingDB files (no writing)

The four raw source filenames are taken from the authors' notebook: `proteins_train_4_org_corrected`, `ligands_train_4_org_corrected` and their test equivalents. We require **one protein per ligand line** and stop if source pairs are misaligned.

In [ ]:
def source_dir(fold, partition):
    prefix = 'train' if partition == 'train' else 'test'
    return DATA_PATH / f'{prefix}_dataset' / f'{prefix}_4_org_{fold}'

def raw_path(fold, partition, kind):
    prefix = 'train' if partition == 'train' else 'test'
    return source_dir(fold, partition) / f'{kind}_{prefix}_4_org_corrected'

source_paths = {f'{fold}/{part}/{kind}': raw_path(fold, part, kind)
                for fold in FOLDS for part in ('train', 'test')
                for kind in ('proteins', 'ligands')}
missing = [str(p) for p in source_paths.values() if not p.is_file()]
if missing:
    raise FileNotFoundError('Missing BindingDB source files (first 12 shown):\n' + '\n'.join(missing[:12]) +
                            f'\nMissing total: {len(missing)}. Check DATA_PATH and extracted fold structure.')

for fold in FOLDS:
    print(f'Fold {fold}:', source_dir(fold, 'train').relative_to(DATA_PATH),
          '|', source_dir(fold, 'test').relative_to(DATA_PATH))

## 2 · Read paired records and examine split sizes

Sequences are normalized consistently (strip whitespace, uppercase). **We do not assume** that the nominal `1042_104` raw files already hold exactly 1,042/104 unique sequences; the original notebook notes a larger Dataset 1 training pool and only 99 original test proteins.

In [ ]:
def read_raw_pairs(fold, part):
    protein_path, ligand_path = raw_path(fold, part, 'proteins'), raw_path(fold, part, 'ligands')
    proteins = ["".join(s.split()).upper() for s in protein_path.read_text().splitlines()]
    ligands = [s.strip().replace(' ', '') for s in ligand_path.read_text().splitlines()]
    if len(proteins) != len(ligands):
        raise ValueError(f'Pair alignment error: {protein_path} has {len(proteins)} rows, '
                         f'{ligand_path} has {len(ligands)} rows')
    if any(not p for p in proteins) or any(not s for s in ligands):
        raise ValueError(f'Blank protein/SMILES in {fold}/{part}: verify input data')
    return pd.DataFrame({'protein_sequence': proteins, 'original_smiles': ligands,
                         'source_fold': fold, 'source_partition': part,
                         'source_row': np.arange(len(proteins), dtype=int)})

raw = {(fold, part): read_raw_pairs(fold, part)
       for fold in FOLDS for part in ('train', 'test')}
summary = pd.DataFrame([
    {'fold': fold, 'partition': part, 'pairs': len(raw[(fold, part)]),
     'unique_proteins': raw[(fold, part)].protein_sequence.nunique(),
     'unique_raw_ligands': raw[(fold, part)].original_smiles.nunique()}
    for fold in FOLDS for part in ('train', 'test')])
display(summary)
print('Original Dataset 1 protein counts:',
      raw[(FOLDS[0], 'train')].protein_sequence.nunique(),
      raw[(FOLDS[0], 'test')].protein_sequence.nunique())

## 3 · Deterministic Dataset 1 protein selection

This is a **documented reconstruction**, not an assertion that the randomly sampled cohort is identical to the authors' original saved cohort. The authors' notebook prioritizes Dataset 1 proteins not in the other training folds, then samples the remainder with seed 17; we retain that principle while enforcing disjoint held-out proteins and stable sorting before sampling.

We first preserve original Dataset 1 test proteins where possible; supplemental test proteins may come from other BindingDB folds, with source provenance retained. Dataset 1 training proteins must have training pairs in its own raw training file. **For an exact published split, prefer a verified supplied split manifest/files and select `reuse_existing`.**

In [ ]:
def stable_choose(candidates, k, rng):
    candidates = sorted(set(candidates))
    if not (0 <= k <= len(candidates)):
        raise ValueError(f'Cannot choose {k} from only {len(candidates)} candidates')
    return sorted(rng.sample(candidates, k))

def reconstruct_split(raw_data, train_n=1042, test_n=104, seed=17):
    rng = random.Random(seed)
    main = FOLDS[0]
    d1_train = set(raw_data[(main, 'train')].protein_sequence)
    d1_test = set(raw_data[(main, 'test')].protein_sequence)
    other_trains = set().union(*(set(raw_data[(f, 'train')].protein_sequence) for f in FOLDS[1:]))
    train_eligible = d1_train - d1_test
    priority = sorted(train_eligible - other_trains)
    if len(priority) > train_n:
        selected_train = stable_choose(priority, train_n, rng)
    else:
        selected_train = sorted(priority) + stable_choose(train_eligible - set(priority),
                                                          train_n - len(priority), rng)
    selected_train = sorted(set(selected_train))
    if len(selected_train) != train_n:
        raise AssertionError('Unexpected training protein count')
    # Original D1 test proteins first; then supplement from other folds.
    first_test = sorted(d1_test - set(selected_train))
    if len(first_test) >= test_n:
        selected_test = stable_choose(first_test, test_n, rng)
    else:
        # Prioritize proteins held out by other folds, then add from their train folds.
        other_tests = set().union(*(set(raw_data[(f, 'test')].protein_sequence) for f in FOLDS[1:]))
        other_all = set().union(*(set(raw_data[(f, p)].protein_sequence)
                                  for f in FOLDS[1:] for p in ('test', 'train')))
        fill1 = stable_choose(other_tests - set(selected_train) - set(first_test),
                              min(test_n - len(first_test), len(other_tests - set(selected_train) - set(first_test))), rng)
        remaining = test_n - len(first_test) - len(fill1)
        fill2 = stable_choose(other_all - set(selected_train) - set(first_test) - set(fill1), remaining, rng)
        selected_test = sorted(set(first_test + fill1 + fill2))
    if len(selected_test) != test_n or set(selected_train) & set(selected_test):
        raise AssertionError('Reconstruction failed count/disjointness check')
    return selected_train, selected_test

def read_existing_split():
    root_tr = source_dir(FOLDS[0], 'train')
    root_te = source_dir(FOLDS[0], 'test')
    a = root_tr / 'd1_tr_unique_proteins.txt'
    b = root_te / 'd1_te_unique_proteins.txt'
    if not a.is_file() or not b.is_file():
        raise FileNotFoundError('Existing unique-protein lists unavailable; use SPLIT_MODE="reconstruct"')
    tr = sorted({''.join(x.split()).upper() for x in a.read_text().splitlines() if x.strip()})
    te = sorted({''.join(x.split()).upper() for x in b.read_text().splitlines() if x.strip()})
    if len(tr) != TRAIN_PROTEINS_TARGET or len(te) != TEST_PROTEINS_TARGET:
        raise ValueError(f'Existing split has {len(tr)} train/{len(te)} test proteins, expected '
                         f'{TRAIN_PROTEINS_TARGET}/{TEST_PROTEINS_TARGET}; do not silently reuse.')
    return tr, te

if SPLIT_MODE == 'reconstruct':
    train_proteins, test_proteins = reconstruct_split(raw, TRAIN_PROTEINS_TARGET,
                                                     TEST_PROTEINS_TARGET, RANDOM_SEED)
else:
    train_proteins, test_proteins = read_existing_split()
assert len(train_proteins) == TRAIN_PROTEINS_TARGET
assert len(test_proteins) == TEST_PROTEINS_TARGET
assert not (set(train_proteins) & set(test_proteins)), 'Training/testing protein overlap!'
print('Selected unique proteins — train:', len(train_proteins), 'test:', len(test_proteins))
print('Original Dataset 1 test proteins preserved:',
      len(set(test_proteins) & set(raw[(FOLDS[0], 'test')].protein_sequence)))

## 4 · Assemble **correctly aligned** protein–ligand pairs

Train rows come from Dataset 1's raw training records. Test rows use the original Dataset 1 test records where available; for supplemental held-out proteins, use the **first available test fold** before considering another fold's training data. Pair-level provenance is saved so this can be audited.

**Caution:** any supplemented test protein/ligand from an earlier training fold is a reconstructed benchmark, not a claim of exact paper reproducibility.

In [ ]:
train_selected = set(train_proteins)
test_selected = set(test_proteins)
train_pairs_raw = raw[(FOLDS[0], 'train')].loc[
    lambda df: df.protein_sequence.isin(train_selected)].copy()

# For each protein select one donor SOURCE; keep ALL rows from that source.
test_sources = [(FOLDS[0], 'test')] + [(f, 'test') for f in FOLDS[1:]] +                [(f, 'train') for f in FOLDS[1:]]
test_donors = {}
for fold, part in test_sources:
    for protein in set(raw[(fold, part)].protein_sequence) & test_selected:
        test_donors.setdefault(protein, (fold, part))
missing_proteins = test_selected - set(test_donors)
if missing_proteins:
    raise ValueError(f'Missing test ligand pairs for {len(missing_proteins)} selected proteins')
parts = []
for (fold, part), proteins_from_source in sorted(
    ((source, {p for p, owner in test_donors.items() if owner == source})
     for source in set(test_donors.values())), key=lambda z: (z[0][0], z[0][1])):
    parts.append(raw[(fold, part)].loc[
        lambda df, proteins_from_source=proteins_from_source: df.protein_sequence.isin(proteins_from_source)].copy())
test_pairs_raw = pd.concat(parts, ignore_index=True)
if set(train_pairs_raw.protein_sequence) != train_selected:
    raise AssertionError('Some selected train proteins have no ligand pairs')
if set(test_pairs_raw.protein_sequence) != test_selected:
    raise AssertionError('Some selected test proteins have no ligand pairs')
print('Raw pair counts: train', len(train_pairs_raw), '| test', len(test_pairs_raw))
print('Test donor fold/partition:')
display(test_pairs_raw.groupby(['source_fold', 'source_partition'])['protein_sequence']
        .agg(['nunique', 'count']).rename(columns={'nunique':'proteins', 'count':'pairs'}))

## 5 · SMILES cleaning with **pair-level filtering**

The authors remove salts by selecting a SMILES component and filter at most 100 atoms. Here we parse RDKit fragments, choose the largest component by heavy-atom count (deterministic ties), canonicalize its SMILES, and then apply the ≤100-atom rule. This **improves chemical identity/leakage checks** but is not byte-for-byte identical to the original notebook's longest-text-fragment heuristic. Invalid SMILES are skipped **together with their matching protein record**.

In [ ]:
def canonical_largest_fragment(smiles, max_atoms=100):
    if not isinstance(smiles, str) or not smiles.strip():
        return None, 'blank_smiles'
    mol = Chem.MolFromSmiles(smiles.strip())
    if mol is None:
        return None, 'invalid_smiles'
    frags = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=True)
    if not frags:
        return None, 'no_fragments'
    ranked = sorted(((f.GetNumHeavyAtoms(), f.GetNumAtoms(), Chem.MolToSmiles(f, canonical=True))
                     for f in frags), key=lambda x: (-x[0], -x[1], x[2]))
    _, atom_count, canonical = ranked[0]
    if atom_count > max_atoms:
        return None, 'over_100_atoms'
    return canonical, None

def clean_pairs(pairs, split):
    rows, reasons = [], Counter()
    for row in pairs.itertuples(index=False):
        cleaned, reason = canonical_largest_fragment(row.original_smiles, MAX_ATOMS)
        if reason:
            reasons[reason] += 1
            continue
        rows.append({'split': split,
                     'protein_sequence': row.protein_sequence,
                     'smiles': cleaned,
                     'original_smiles': row.original_smiles,
                     'source_fold': row.source_fold,
                     'source_partition': row.source_partition,
                     'source_row': int(row.source_row)})
    return pd.DataFrame(rows), dict(reasons)

train_pairs, train_rejections = clean_pairs(train_pairs_raw, 'train')
test_pairs, test_rejections = clean_pairs(test_pairs_raw, 'test')
print('Cleaned pairs: train', len(train_pairs), 'test', len(test_pairs))
print('Rejected train:', train_rejections, '| rejected test:', test_rejections)
if train_pairs.empty or test_pairs.empty:
    raise ValueError('One split became empty after filtering')
if set(train_pairs.protein_sequence) & set(test_pairs.protein_sequence):
    raise AssertionError('Protein leakage between train and test')
if train_pairs.protein_sequence.nunique() != TRAIN_PROTEINS_TARGET:
    raise ValueError('Filtering removed all ligands for one or more selected training proteins')
if test_pairs.protein_sequence.nunique() != TEST_PROTEINS_TARGET:
    raise ValueError('Filtering removed all ligands for one or more selected held-out proteins')
print('Unique canonical ligand counts:', train_pairs.smiles.nunique(), test_pairs.smiles.nunique())
print('Canonical ligand intersection between splits:',
      len(set(train_pairs.smiles) & set(test_pairs.smiles)),
      '(protein split disjointness is separately enforced)')

## 6 · Atom vocabulary and BindingDB exclusion set

The paper describes a 65-atom-type BindingDB vocabulary; the actual count may vary with filtering and ChEMBL preprocessing. We save atom features as the authors did (`symbol,formal_charge,explicit_Hs`) in stable sorted order. Build a **canonical BindingDB exclusion set across all five folds**, rather than only the final Dataset 1 rows, for optional ChEMBL anti-leakage preprocessing.

In [ ]:
def atom_type_set(smiles_iter):
    atoms = set()
    for s in smiles_iter:
        mol = Chem.MolFromSmiles(s)
        if mol is None:
            raise ValueError(f'Unexpected invalid cleaned SMILES: {s}')
        for atom in mol.GetAtoms():
            atoms.add(f'{atom.GetSymbol()},{atom.GetFormalCharge()},{atom.GetNumExplicitHs()}')
    return atoms

atom_types = atom_type_set(set(train_pairs.smiles) | set(test_pairs.smiles))
all_bindingdb_smiles = set()
all_rejects = Counter()
for df in raw.values():
    for smiles in df.original_smiles:
        normalized, why = canonical_largest_fragment(smiles, MAX_ATOMS)
        if why:
            all_rejects[why] += 1
        else:
            all_bindingdb_smiles.add(normalized)
print('Atom types in cleaned Dataset 1 pairs:', len(atom_types))
print('Canonical ligand identities in all 5 folds:', len(all_bindingdb_smiles))
print('Source molecules rejected during exclusion-set construction:', dict(all_rejects))

## 7 · Write reproducible BindingDB artifacts (no source overwrite)

**Outputs:** `train_pairs.tsv`, `test_pairs.tsv` (row-aligned protein/ligand pairs), `train_proteins.txt`, `test_proteins.txt`, `atom_types.txt`, `bindingdb_exclusion_smiles.smi`, `train_ligands.smi`, `test_ligands.smi`, `dataset_manifest.json`.

This intentionally does **not** impersonate the original legacy `d1_tr_cdgcn.txt` format until the downstream MXNet data-loader/schema is verified. The supplied paper does not define that file format. The notebook does not create an Ames-labelled pretraining CSV.

In [ ]:
def sha256_file(path, chunk_size=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()

def refuse_overwrite(paths):
    existing = [str(p) for p in paths if p.exists()]
    if existing and not ALLOW_OVERWRITE:
        raise FileExistsError('Output exists; not overwriting. Change OUTPUT_DIR (recommended) or '
                              f'explicitly set ALLOW_OVERWRITE=True. Existing: {existing[:8]}')

def save_lines(path, items):
    with path.open('w') as f:
        for item in items:
            f.write(str(item) + '\n')

artifact_names = ['train_pairs.tsv', 'test_pairs.tsv', 'train_proteins.txt',
                  'test_proteins.txt', 'atom_types.txt', 'bindingdb_exclusion_smiles.smi',
                  'train_ligands.smi', 'test_ligands.smi', 'dataset_manifest.json']
artifacts = {name: OUTPUT_DIR / name for name in artifact_names}
refuse_overwrite(artifacts.values())
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
train_pairs.to_csv(artifacts['train_pairs.tsv'], sep='\t', index=False)
test_pairs.to_csv(artifacts['test_pairs.tsv'], sep='\t', index=False)
save_lines(artifacts['train_proteins.txt'], sorted(train_proteins))
save_lines(artifacts['test_proteins.txt'], sorted(test_proteins))
save_lines(artifacts['atom_types.txt'], sorted(atom_types))
save_lines(artifacts['bindingdb_exclusion_smiles.smi'], sorted(all_bindingdb_smiles))
save_lines(artifacts['train_ligands.smi'], train_pairs.smiles)
save_lines(artifacts['test_ligands.smi'], test_pairs.smiles)
manifest = {
    'purpose': 'CTDDG BindingDB pre-fine-tuning preparation (unlabelled)',
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'repo_root': str(REPO_ROOT), 'data_path': str(DATA_PATH),
    'source_folds': list(FOLDS), 'split_mode': SPLIT_MODE,
    'random_seed': RANDOM_SEED, 'max_atoms': MAX_ATOMS,
    'salt_policy': 'largest RDKit fragment by heavy atom count; canonical SMILES',
    'train_proteins': int(train_pairs.protein_sequence.nunique()),
    'test_proteins': int(test_pairs.protein_sequence.nunique()),
    'train_pairs': len(train_pairs), 'test_pairs': len(test_pairs),
    'train_rejections': train_rejections, 'test_rejections': test_rejections,
    'atom_types': len(atom_types),
    'test_donor_counts': {f'{f}/{p}': sum(source == (f, p) for source in test_donors.values())
                          for f, p in test_sources if any(source == (f, p) for source in test_donors.values())},
    'source_sha256': {key: sha256_file(path) for key, path in sorted(source_paths.items())},
    'output_sha256': {name: sha256_file(path) for name, path in artifacts.items()
                      if name != 'dataset_manifest.json'},
    'warnings': [
        'Not paper-identical without the verified original protein split manifest.',
        'Ames toxicity labels and protein embeddings are not yet included.',
        'Do not train CTDDG directly from this TSV without verifying MXNet loader inputs.',
        'Raw ChEMBL preprocessing is separate and optional.'
    ]
}
artifacts['dataset_manifest.json'].write_text(json.dumps(manifest, indent=2) + '\n')
print('Saved verified BindingDB preparation outputs to:', OUTPUT_DIR)
print('Manifest:', artifacts['dataset_manifest.json'])

## 8 · Optional ChEMBL molecule preprocessing (**OFF** by default)

The paper pretrains CTDDG on ChEMBL molecules with Ames mutagenicity labels predicted using a trained Random Forest classifier (paper §2.4). **This cell only creates an unlabeled, filtered, deduplicated, BindingDB-excluded SMILES list. It does not infer toxicity labels.**

Input: `datasets/data/chembl/chembl.csv` with a `Smiles` column (comma or semicolon delimited). One million+ structures can consume time and memory; run it as an approved batch job/compute allocation if required. Existing original ChEMBL label files/checkpoints are preserved.

In [ ]:
if RUN_CHEMBL_PREP:
    if not CHEMBL_CSV.is_file():
        raise FileNotFoundError(f'ChEMBL source not found: {CHEMBL_CSV}')
    out_chembl = OUTPUT_DIR / 'chembl_filtered_no_bindingdb_unlabelled.smi'
    out_chembl_stats = OUTPUT_DIR / 'chembl_preprocessing_stats.json'
    refuse_overwrite([out_chembl, out_chembl_stats])
    # Detect separators and column names without reading the whole data file.
    header = CHEMBL_CSV.open().readline()
    sep = ';' if header.count(';') > header.count(',') else ','
    header_columns = pd.read_csv(CHEMBL_CSV, sep=sep, nrows=0).columns
    smiles_col = next((x for x in header_columns if x.strip().lower() == 'smiles'), None)
    if smiles_col is None:
        raise ValueError(f'No Smiles column in {CHEMBL_CSV}; columns={list(header_columns)}')
    seen = set()
    stats = Counter()
    for chunk in pd.read_csv(CHEMBL_CSV, sep=sep, usecols=[smiles_col], chunksize=25000):
        for original in chunk[smiles_col]:
            stats['read_rows'] += 1
            if not isinstance(original, str):
                stats['null_smiles'] += 1
                continue
            cleaned, reason = canonical_largest_fragment(original, MAX_ATOMS)
            if reason:
                stats[reason] += 1
            elif cleaned in all_bindingdb_smiles:
                stats['bindingdb_overlap_excluded'] += 1
            elif cleaned in seen:
                stats['duplicate_chembl_excluded'] += 1
            else:
                seen.add(cleaned)
    save_lines(out_chembl, sorted(seen))
    stats['saved_unlabelled_molecules'] = len(seen)
    stats['chembl_input_sha256'] = sha256_file(CHEMBL_CSV)
    out_chembl_stats.write_text(json.dumps(dict(stats), indent=2) + '\n')
    print('Saved', len(seen), 'unlabelled ChEMBL structures:', out_chembl)
else:
    print('ChEMBL preprocessing skipped (RUN_CHEMBL_PREP=False)')

## 9 · Optional ProtTransBERT-BFD embeddings (**OFF** by default)

The original `data_preprocessing_1.ipynb` uses `ProtTransBertBFDEmbedder.embed()`. In default `EMBEDDING_MODE="residue"`, this section retains **per-residue** arrays, without silently pooling them. Switch to `"mean"` only after verifying that the MXNet loader expects a fixed-dimensional protein vector; this changes the feature representation.

Embeddings are cached one protein at a time in a gzip-compressed HDF5 file. Unique sequences have stable SHA-256 keys; provenance and embedding dimensions are recorded. Embedding requires `bio_embeddings` and `h5py`, and can be very expensive: **use an allocated GPU node, not the Bhavani login node**. No model training is started here.

In [ ]:
if RUN_EMBEDDINGS:
    try:
        import h5py
        from bio_embeddings.embed import ProtTransBertBFDEmbedder
    except ImportError as exc:
        raise ImportError('Install compatible bio-embeddings and h5py in the allocated '
                          'GPU environment before enabling embeddings') from exc
    embed_file = OUTPUT_DIR / f'prottransbert_bfd_{EMBEDDING_MODE}.h5'
    index_file = OUTPUT_DIR / f'prottransbert_bfd_{EMBEDDING_MODE}_index.tsv'
    refuse_overwrite([embed_file, index_file])
    all_proteins = sorted(set(train_proteins) | set(test_proteins))
    embedder = ProtTransBertBFDEmbedder()
    embed_index = []
    with h5py.File(embed_file, 'x' if not ALLOW_OVERWRITE else 'w') as h5:
        h5.attrs['model'] = 'ProtTransBertBFDEmbedder'
        h5.attrs['representation'] = EMBEDDING_MODE
        for ix, seq in enumerate(all_proteins, 1):
            key = hashlib.sha256(seq.encode()).hexdigest()
            arr = np.asarray(embedder.embed(seq), dtype=np.float32)
            if arr.ndim != 2 or arr.shape[0] == 0:
                raise ValueError(f'Unexpected embedding shape for protein {ix}: {arr.shape}')
            if EMBEDDING_MODE == 'mean':
                arr = arr.mean(axis=0)  # only after user explicitly selects this mode
            if not np.isfinite(arr).all():
                raise ValueError(f'Non-finite embeddings for protein {ix}')
            h5.create_dataset(key, data=arr, compression='gzip', compression_opts=3)
            embed_index.append({'sha256': key, 'protein_sequence': seq,
                                'shape': str(tuple(arr.shape)),
                                'split': 'train' if seq in set(train_proteins) else 'test'})
            if ix % 50 == 0 or ix == len(all_proteins):
                print('Embedded', ix, '/', len(all_proteins))
    pd.DataFrame(embed_index).to_csv(index_file, sep='\t', index=False)
    print('Embeddings saved:', embed_file)
    print('Before CTDDG training, verify shape, representation, and serialization '
          'against the exact MXNet dataset loader.')
else:
    print('Protein embedding skipped (RUN_EMBEDDINGS=False)')

## 10 · Final quality report and next CTDDG steps

**Readiness checklist before training:**

1. Confirm the split matches the intended experimental cohort and protein similarity constraints (the original notebook's bulk EMBOSS `needle` loops are **not** executed here; they are extremely expensive and contain bugs).
2. Check the exact CTDDG MXNet loader's protein-embedding shape and file format before generating legacy CDGCN training text.
3. Verify provenance of ChEMBL Random Forest Ames labels and BindingDB pkCSM Ames labels. *Do not substitute an unverified* `chembl_with_ames_predictions.csv` *for a labelled pretraining file*.
4. Preserve manifests and source hashes. Run the preflight and objective diagnostics before costly pretraining/fine-tuning.

Paper references: Singh, Bhadra & Bhadra, *CTDDG: Graph-Based Deep Generative Model for Low-Toxic Drug Design for a Target Protein with Optimal Binding Affinity*, §§2.1–2.4. Original source: [CTDDG/code](https://github.com/sahelybhadra/CTDDG/tree/main/code).

In [ ]:
report = {
    'output_dir': str(OUTPUT_DIR),
    'train_pairs': len(train_pairs),
    'test_pairs': len(test_pairs),
    'unique_train_proteins': train_pairs.protein_sequence.nunique(),
    'unique_test_proteins': test_pairs.protein_sequence.nunique(),
    'protein_overlap': len(set(train_pairs.protein_sequence) & set(test_pairs.protein_sequence)),
    'canonical_ligand_overlap': len(set(train_pairs.smiles) & set(test_pairs.smiles)),
    'atom_types': len(atom_types),
    'pretraining_labels_ready': False,
    'bindingdb_pkcsm_labels_ready': False,
    'model_loader_contract_verified': False,
}
print(json.dumps(report, indent=2))